# PRE-TRAINED CLIP FEW SHOT CLASSIFICATION


---
In questo primo codice vogliamo caricare il modello CLIP pre-addestrato, e testarlo (senza fire fine-tuning) sulle immagini di test. Questo per avere una prima idea delle performance e capire come muoverci.

Per altre info andare al seguente link: https://github.com/openai/CLIP

In [ ]:
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

In [2]:
import torch
import clip
from PIL import Image

In [4]:
# Utilizziamo la GPU se disponibile
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo utilizzato per i calcoli: {device}")

Dispositivo utilizzato per i calcoli: cuda


This method returns the names of the available CLIP models.



In [6]:
print("Ecco i modelli pre-trainati disponibili:\n")
list_models = clip.available_models()
for i in range(len(list_models)):
  print(list_models[i])

Ecco i modelli pre-trainati disponibili:

RN50
RN101
RN50x4
RN50x16
RN50x64
ViT-B/32
ViT-B/16
ViT-L/14
ViT-L/14@336px


This method returns the model and the TorchVision transform needed by the model, specified by the model name returned by clip.available_models(). It will download the model as necessary. The name argument can also be a path to a local checkpoint.

In [8]:
model, preprocess = clip.load("ViT-B/32", device=device) # Carichiamo il modello e la relativa Transform necessaria (utilizzando la GPU)
print(f"Assieme al model, questo metodo ritorna in output anche preprocess, di questo tipo {type(preprocess)}")

Assieme al model, questo metodo ritorna in output anche preprocess, di questo tipo <class 'torchvision.transforms.transforms.Compose'>


Adesso che ho il modello, voglio preparare i path per le mie immagini di test, così da poterle classificare

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import os
TRAINING_DATA_PATH = os.path.join('/content','drive','MyDrive','AdvancedML','Project','release','images')

This method returns a LongTensor containing tokenized sequences of given text input(s). This can be used as the input to the model

In [13]:
text = clip.tokenize(["a diagram", "a dog", "a cat"]).to(device)
text.shape

torch.Size([3, 77])

In [ ]:
img = None # Immagine da far testare al modello

# Disattivo il salvataggio dei gradienti perchè sono in inference mode
with torch.no_grad():

    #Given a batch of images, returns the image features encoded by the vision portion of the CLIP model.
    image_features = model.encode_image(img)

    #Given a batch of text tokens, returns the text features encoded by the language portion of the CLIP model.
    text_features = model.encode_text(text)

    # Given a batch of images and a batch of text tokens, returns two Tensors, containing the logit scores corresponding to each image and text input. The values are cosine similarities between the corresponding image and text features, times 100.
    logits_per_image, logits_per_text = model(img, text)

    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:",probs)

Label probs: [[3.290e-04 9.966e-01 3.271e-03]]
